# Sprint 2: Entrenamiento con HDF5 (rápido y estable)

**Flujo de cada sesión:**
1. Copia `nih_images.h5` de Drive a `/content/` (~2-3 min).
2. Copia los CSVs de splits.
3. Instala los módulos del Sprint 2.
4. Entrena (~15-30 min en T4).

Sin errores de Drive I/O. Sin copiar 25k archivos.
**Runtime:** T4 GPU.


## 1. Montar Drive y copiar HDF5 a disco local

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'
CODE_DIR     = f'{PROJECT_ROOT}/code'

import os, shutil, time, sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Copiar HDF5 a disco local (mucho más rápido que leer desde Drive)
H5_DRIVE = f'{NIH_DIR}/nih_images.h5'
H5_LOCAL = '/content/nih_images.h5'

if os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 1_000_000_000:
    size_gb = os.path.getsize(H5_LOCAL) / (1024**3)
    print(f'✓ HDF5 ya en disco local ({size_gb:.2f} GB)')
else:
    size_gb = os.path.getsize(H5_DRIVE) / (1024**3)
    print(f'Copiando HDF5 a disco local ({size_gb:.2f} GB)...')
    t0 = time.time()
    shutil.copy2(H5_DRIVE, H5_LOCAL)
    elapsed = time.time() - t0
    print(f'✓ Copiado en {elapsed/60:.1f} min')


## 2. Copiar CSVs de splits a disco local

In [ ]:
LOCAL_PROCESSED = '/content/processed_s2'
os.makedirs(LOCAL_PROCESSED, exist_ok=True)

for csv in ['train.csv', 'val.csv', 'test.csv']:
    src = f'{PROJECT_ROOT}/data/processed_s2/{csv}'
    dst = f'{LOCAL_PROCESSED}/{csv}'
    if not os.path.exists(dst):
        shutil.copy(src, dst)
    print(f'✓ {csv}: {len(__import__("pandas").read_csv(dst))} filas')


In [ ]:
!pip install -q torchxrayvision scipy h5py
print('deps OK')


## 3. Instalar módulos del Sprint 2

In [ ]:
from pathlib import Path

# nih_hdf5.py
content = r'''"""
src/datasets/nih_hdf5.py

Dataset de PyTorch que lee imágenes desde un archivo HDF5.
Reemplaza nih.py para entrenamiento en Colab sin errores de Drive I/O.

Ventajas sobre leer PNGs sueltos desde Drive:
  - Un solo archivo → copia rápida a /content/ al inicio de sesión.
  - Lectura 10-20x más rápida (acceso secuencial vs random a 25k archivos).
  - Compatible con num_workers > 0 (h5py soporta acceso paralelo).
  - Sin OSError Errno 5 de Drive.
"""
from __future__ import annotations

from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


class NIHDatasetHDF5(Dataset):
    """
    Dataset NIH que lee imágenes desde un archivo HDF5.

    El HDF5 tiene la estructura:
      /image_names: array de strings con nombres de archivo
      /images/{nombre.png}: array uint8 (H, W) en escala de grises

    Args:
        csv_path: CSV con columnas Image Index, label, Patient ID, etc.
        h5_path: ruta al archivo nih_images.h5.
        transform: pipeline de preprocesamiento (de transforms.py).
    """

    def __init__(self, csv_path: str | Path, h5_path: str | Path,
                 transform=None):
        self.csv_path = Path(csv_path)
        self.h5_path  = Path(h5_path)
        self.transform = transform

        if not self.csv_path.exists():
            raise FileNotFoundError(f"CSV no encontrado: {self.csv_path}")
        if not self.h5_path.exists():
            raise FileNotFoundError(
                f"HDF5 no encontrado: {self.h5_path}\n"
                "Corre primero S2_05_create_hdf5.ipynb"
            )

        self.df = pd.read_csv(self.csv_path)
        self._validate_columns()

        # Abrir HDF5 en modo lectura (se mantiene abierto durante el entrenamiento)
        # NOTA: h5py con swmr=False es thread-safe para lectura.
        self._h5 = h5py.File(self.h5_path, 'r')

    def _validate_columns(self) -> None:
        required = {'Image Index', 'label', 'Patient ID'}
        missing = required - set(self.df.columns)
        if missing:
            raise ValueError(f"Columnas faltantes: {missing}")

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        image_name = row['Image Index']
        label = int(row['label'])

        # Leer desde HDF5 (mucho más rápido que abrir PNG desde Drive)
        arr = self._h5['images'][image_name][:]  # (H, W) uint8

        if self.transform is not None:
            arr = self.transform(arr)

        return {
            'image': arr,
            'label': torch.tensor(label, dtype=torch.long),
            'image_index': image_name,
        }

    def class_distribution(self) -> dict:
        counts = self.df['label'].value_counts().to_dict()
        return {int(k): int(v) for k, v in counts.items()}

    def __del__(self):
        # Cerrar el archivo HDF5 al destruir el objeto
        if hasattr(self, '_h5') and self._h5.id.valid:
            self._h5.close()


def build_nih_hdf5_datasets(processed_dir: str, h5_path: str,
                             train_transform=None,
                             eval_transform=None) -> tuple:
    """Crea los 3 datasets (train/val/test) leyendo desde HDF5."""
    processed = Path(processed_dir)
    h5 = Path(h5_path)
    train_ds = NIHDatasetHDF5(processed / "train.csv", h5, train_transform)
    val_ds   = NIHDatasetHDF5(processed / "val.csv",   h5, eval_transform)
    test_ds  = NIHDatasetHDF5(processed / "test.csv",  h5, eval_transform)
    return train_ds, val_ds, test_ds
'''
path = f'{CODE_DIR}/src/datasets/nih_hdf5.py'
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path}')


In [ ]:
# run_train_s2_hdf5.py
content = r'''"""
scripts/run_train_s2_hdf5.py

Entrypoint Sprint 2 usando HDF5 para lectura rápida de imágenes.
"""
from __future__ import annotations
import json, sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.datasets.nih_hdf5 import build_nih_hdf5_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.models.baseline import build_baseline_model
from src.training.train_s2 import train_model_s2
from src.training.utils import get_device, get_logger, load_config, set_seed


def main(config_path=None, h5_path=None, processed_dir=None):
    if config_path is None:
        config_path = CODE_DIR / "configs" / "sprint2.yaml"
    cfg = load_config(config_path)
    set_seed(cfg.project['seed'])

    log_file = Path(cfg.paths['logs']) / "train_s2.log"
    logger = get_logger("train_s2", log_file=log_file)

    logger.info("=" * 70)
    logger.info("RUN TRAIN S2 (HDF5) — 4 clases, finetune full")
    logger.info("=" * 70)

    device = get_device()
    logger.info("Device: %s | GPU: %s", device,
                torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU')

    # Rutas (pueden venir de argumentos o del config)
    h5  = h5_path      or '/content/nih_images.h5'
    prc = processed_dir or cfg.paths['data_processed']

    train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])
    eval_tfm  = build_eval_transform(image_size=cfg.preprocessing['image_size'])

    train_ds, val_ds, test_ds = build_nih_hdf5_datasets(prc, h5, train_tfm, eval_tfm)
    logger.info("Train: %d | Val: %d | Test: %d", len(train_ds), len(val_ds), len(test_ds))

    # num_workers=2 funciona bien con HDF5 (a diferencia de Drive directo)
    nw = cfg.training.get('num_workers', 2)
    train_loader = DataLoader(train_ds, batch_size=cfg.training['batch_size'],
                              shuffle=True, num_workers=nw,
                              pin_memory=(device.type=='cuda'))
    val_loader   = DataLoader(val_ds, batch_size=cfg.training['batch_size'],
                              shuffle=False, num_workers=nw,
                              pin_memory=(device.type=='cuda'))

    model = build_baseline_model(cfg).to(device)
    logger.info("Modelo: %s | mode=%s | num_classes=%d | trainable=%d",
                model.__class__.__name__, cfg.model['finetune_mode'],
                cfg.model['num_classes'], model.trainable_params())

    result = train_model_s2(model, train_loader, val_loader, cfg, device, logger)

    history_path = Path(cfg.paths['experiments']) / "history_s2.json"
    history_path.parent.mkdir(parents=True, exist_ok=True)
    with history_path.open('w') as f:
        json.dump(result, f, indent=2)

    logger.info("RESUMEN: best epoch=%d | best val AUC_macro=%.4f",
                result['best_epoch'], result['best_val_auc_macro'])


if __name__ == "__main__":
    main()
'''
path = f'{CODE_DIR}/scripts/run_train_s2_hdf5.py'
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path}')


## 4. Smoke test: leer una imagen del HDF5

In [ ]:
import h5py
import matplotlib.pyplot as plt

for mod in list(__import__('sys').modules.keys()):
    if mod.startswith('src.'): del __import__('sys').modules[mod]

from src.datasets.nih_hdf5 import build_nih_hdf5_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.training.utils import load_config

cfg = load_config(f'{CODE_DIR}/configs/sprint2.yaml')
train_tfm = build_train_transform(image_size=224)
eval_tfm  = build_eval_transform(image_size=224)

train_ds, val_ds, test_ds = build_nih_hdf5_datasets(
    LOCAL_PROCESSED, H5_LOCAL, train_tfm, eval_tfm
)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Distribución train: {train_ds.class_distribution()}')

# Leer una muestra
sample = train_ds[0]
img = sample['image'].squeeze().numpy()
print(f'\nMuestra: shape={sample["image"].shape} | range=[{img.min():.0f}, {img.max():.0f}]')
print(f'Label: {sample["label"].item()} | File: {sample["image_index"]}')

plt.figure(figsize=(4,4))
plt.imshow(img, cmap='gray')
plt.title(f'label={sample["label"].item()} | {sample["image_index"]}')
plt.axis('off')
plt.show()


## 5. Entrenamiento Sprint 2

Tiempo estimado: **15-30 min** en T4 GPU.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/run_train_s2_hdf5.py'],
    capture_output=True, text=True,
    env={**__import__('os').environ,
         'LOCAL_PROCESSED': LOCAL_PROCESSED,
         'H5_LOCAL': H5_LOCAL},
)
print('STDOUT:')
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print()
print('LOG:')
print(result.stderr[-6000:] if len(result.stderr) > 6000 else result.stderr)


## 6. Curvas de entrenamiento

In [ ]:
import json, matplotlib.pyplot as plt

history_path = f'{PROJECT_ROOT}/experiments_s2/history_s2.json'
with open(history_path) as f:
    data = json.load(f)

history = data['history']
epochs = [h['epoch'] for h in history]
CLASS_NAMES = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}

print(f"Best epoch: {data['best_epoch']} | Best val AUC_macro: {data['best_val_auc_macro']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (tr_key, va_key, title) in zip(axes, [
    ('train_loss', 'val_loss', 'Loss'),
    (None, None, 'AUC por clase'),
    (None, None, 'AUC Macro'),
]):
    if title == 'Loss':
        ax.plot(epochs, [h['train_loss'] for h in history], 'o-', label='train', color='steelblue')
        ax.plot(epochs, [h['val_loss']   for h in history], 's-', label='val',   color='coral')
        ax.set_title('Loss'); ax.set_ylabel('Loss')
    elif title == 'AUC por clase':
        colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']
        for i in range(4):
            aucs = [h['val_metrics']['auc_per_class'][i] for h in history]
            ax.plot(epochs, aucs, 'o-', label=CLASS_NAMES[i], color=colors[i])
        ax.set_title('Val AUC por clase'); ax.set_ylim(0, 1.05)
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4)
    else:
        ax.plot(epochs, [h['train_metrics']['auc_macro'] for h in history], 'o-', label='train', color='steelblue')
        ax.plot(epochs, [h['val_metrics']['auc_macro']   for h in history], 's-', label='val',   color='coral')
        ax.set_title('AUC Macro'); ax.set_ylim(0, 1.05)
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4)

    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle(f"Sprint 2 — best val AUC_macro={data['best_val_auc_macro']:.4f} @ epoch {data['best_epoch']}")
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments_s2/figures/training_curves_s2.png', dpi=120)
plt.show()
